# F1 Podium Predictor — Complete Learning Guide

This notebook walks you through building a machine learning model that predicts which three drivers will finish on the F1 podium, using historical results, driving style data, and car performance signals.

---

## What you will build

A **binary classifier** that answers: *for a given driver in a given race, what is the probability they finish in the top 3?*

You run this for all 20 drivers before a race, then take the 3 highest probabilities as your predicted podium.

---

## Project structure

```
f1_podium_features.py    ← data collection + feature engineering
f1_podium_model.py       ← model training + evaluation
f1_podium_dashboard.py   ← Streamlit prediction UI
f1_podium_guide.ipynb    ← this learning notebook
```

---

## Three data sources, three feature categories

| Source | What it gives you | Feature category |
|--------|------------------|------------------|
| Jolpica API | Race results, qualifying, constructor standings | Historical form + car quality |
| FastF1 | Lap times, telemetry, tyre data | Driving style proxies |
| Derived | Computed from the above | Quali gap to pole, rolling averages |

---

## Install everything first

```bash
pip install fastf1 pandas numpy scikit-learn xgboost matplotlib plotly requests joblib streamlit
```

---
# Part 1 — Data collection

## Step 1.1 — Fetch race results from Jolpica

Jolpica is the free, open-source replacement for the now-defunct Ergast API. It gives you every race result going back to 1950 via simple REST calls.

For each race we collect:
- Finishing position (our target comes from this)
- Grid position (starting position on race day)
- Points scored
- Race status (Finished / Retired / Accident — needed for DNF rate)
- Constructor (team)

> **Why 2018 onwards?** Regulations changed significantly in 2017 (aero rules) and 2022 (ground effect). Training across major regulation changes risks the model learning patterns that no longer apply. Starting from 2018 gives a good balance of data volume vs relevance.

In [ ]:
import requests
import pandas as pd
import numpy as np
import time

JOLPICA_BASE = "https://api.jolpi.ca/ergast/f1"

def jolpica_get(path):
    """Fetch one page of Jolpica JSON."""
    url = f"{JOLPICA_BASE}/{path}.json?limit=100"
    r = requests.get(url, timeout=10)
    r.raise_for_status()
    return r.json().get("MRData", {})

# Fetch 2023 season results as a worked example
data  = jolpica_get("2023/results")
races = data.get("RaceTable", {}).get("Races", [])

rows = []
for race in races:
    for res in race.get("Results", []):
        rows.append({
            "season":     2023,
            "round":      int(race["round"]),
            "race_name":  race["raceName"],
            "driver_id":  res["Driver"]["driverId"],
            "constructor": res["Constructor"]["constructorId"],
            "grid":       int(res.get("grid", 0)),
            "position":   int(res["position"]) if res.get("position","").isdigit() else 99,
            "points":     float(res.get("points", 0)),
            "status":     res.get("status", ""),
            "on_podium":  1 if res.get("position","").isdigit() and int(res["position"]) <= 3 else 0,
        })

results_2023 = pd.DataFrame(rows)
print(f"Rows: {len(results_2023)}")
print(f"Podium rate: {results_2023['on_podium'].mean()*100:.1f}%  (should be ~15%)")
results_2023.head(10)

## Step 1.2 — Fetch qualifying results

Qualifying tells you where each driver starts the race. It's the **single strongest predictor** of podium likelihood — roughly 60–70% of F1 podiums come from the top 4 grid positions.

We also compute **gap to pole** (qualifying time minus fastest Q3 time). This is more informative than grid position alone because it captures *how much* slower a driver was, not just their relative rank.

In [ ]:
data  = jolpica_get("2023/qualifying")
races = data.get("RaceTable", {}).get("Races", [])

quali_rows = []
for race in races:
    for res in race.get("QualifyingResults", []):
        # Use best available time: Q3 > Q2 > Q1
        q_time = res.get("Q3", res.get("Q2", res.get("Q1", None)))
        quali_rows.append({
            "season":    2023,
            "round":     int(race["round"]),
            "driver_id": res["Driver"]["driverId"],
            "quali_pos": int(res["position"]),
            "q_time":    q_time,
        })

quali_2023 = pd.DataFrame(quali_rows)

# Convert time string "1:30.123" → seconds
def parse_time(t):
    if not t or not isinstance(t, str):
        return None
    try:
        m, s = t.split(":")
        return float(m) * 60 + float(s)
    except:
        return None

quali_2023["q_sec"] = quali_2023["q_time"].apply(parse_time)

# Compute gap to pole for each race
pole_times = quali_2023[quali_2023["quali_pos"] == 1].set_index(["season","round"])["q_sec"]
quali_2023 = quali_2023.join(pole_times.rename("pole_time"), on=["season","round"])
quali_2023["quali_gap"] = (quali_2023["q_sec"] - quali_2023["pole_time"]).fillna(3.0)

print("Sample qualifying data with gap to pole:")
quali_2023.head(10)

---
# Part 2 — Feature engineering

## Step 2.1 — Rolling historical form

Instead of using a driver's all-time stats, we use **rolling windows** over their last 5 races. This captures current form — a driver on a hot streak is more valuable signal than their career average.

We `.shift(1)` before the rolling — this is critical. It prevents **data leakage**, where the model sees the current race's result when computing the feature for that same race. The shift ensures we only look *backward*, never at the current row.

**Features created:**
- `rolling_podium_rate` — what fraction of recent races ended on the podium?
- `rolling_win_rate` — what fraction ended with a race win?
- `rolling_points_avg` — average points per race (captures consistent scoring)
- `rolling_dnf_rate` — reliability signal (Did Not Finish rate)
- `rolling_grid_avg` — average qualifying position (form proxy)

In [ ]:
ROLLING_N = 5  # look back this many races

# Sort chronologically — essential for rolling windows to work correctly
results_2023 = results_2023.sort_values(["driver_id", "round"]).copy()

# Group by driver, then apply rolling calculations
# shift(1) moves each value forward by 1 row — the race's feature
# is computed from the PREVIOUS race, not the current one
g = results_2023.groupby("driver_id")

results_2023["rolling_podium_rate"] = (
    g["on_podium"].transform(lambda x: x.shift(1).rolling(ROLLING_N, min_periods=1).mean())
)
results_2023["rolling_win_rate"] = (
    g["position"].transform(lambda x: x.shift(1).rolling(ROLLING_N, min_periods=1).apply(lambda s: (s==1).mean()))
)
results_2023["rolling_points_avg"] = (
    g["points"].transform(lambda x: x.shift(1).rolling(ROLLING_N, min_periods=1).mean())
)
results_2023["rolling_dnf_rate"] = (
    g["status"].transform(
        lambda x: x.shift(1).rolling(ROLLING_N, min_periods=1)
                   .apply(lambda s: s.str.contains("Retired|Accident|Engine", na=False).mean())
    )
)

print("Rolling form features added:")
results_2023[["driver_id","round","on_podium",
              "rolling_podium_rate","rolling_win_rate","rolling_points_avg"]].head(20)

## Step 2.2 — Car performance features

A driver's podium chances are heavily influenced by their car. A midfield driver in the best car on the grid has fundamentally different odds than a champion in a slower car.

We proxy car quality with **constructor championship position** — position 1 is the fastest car, position 10 the slowest. We get this from Jolpica's constructor standings endpoint.

> **Why not use official horsepower or downforce figures?** Teams don't publish these. Constructor championship position captures the aggregate effect of all car performance factors (power unit, aerodynamics, reliability) through actual race results — which is exactly the information the model needs.

In [ ]:
data      = jolpica_get("2023/constructorStandings")
standings = data.get("StandingsTable", {}).get("StandingsLists", [])

car_rows = []
for s in standings:
    for entry in s.get("ConstructorStandings", []):
        car_rows.append({
            "constructor": entry["Constructor"]["constructorId"],
            "car_rank":    int(entry["position"]),      # 1=fastest, 10=slowest
            "car_points":  float(entry.get("points", 0)),
        })

car_df = pd.DataFrame(car_rows).drop_duplicates("constructor")

# Merge car quality into results
results_2023 = results_2023.merge(car_df, on="constructor", how="left")
results_2023["car_rank"] = results_2023["car_rank"].fillna(6)   # midfield default

print("Car rankings:")
print(car_df.sort_values("car_rank").to_string(index=False))

## Step 2.3 — Driving style features from FastF1

This is the most distinctive part of the feature set. We load lap telemetry from FastF1 and derive *style proxies* — numerical summaries of how aggressively or conservatively a driver operates.

**Features derived:**
- `tyre_deg_slope` — how fast does their lap time increase per lap of tyre age? Aggressive drivers wear tyres faster (steeper slope). Smooth drivers manage tyres better (shallower slope).
- `avg_throttle_pct` — average throttle input across a race stint. Higher = more aggressive acceleration style.
- `overtake_count` — positions gained from grid to finish. High count = strong racecraft, ability to pass.

> **Important:** FastF1 is slow to load (~30 seconds per session). Always enable caching.

In [ ]:
import fastf1
fastf1.Cache.enable_cache("f1_cache")

# Load a single race — Bahrain 2023 as example
session = fastf1.get_session(2023, 1, "R")
session.load(telemetry=True, weather=False)
laps = session.laps

style_rows = []
for driver in laps["DriverNumber"].unique():
    drv_laps = laps[laps["DriverNumber"] == driver].copy()
    drv_laps = drv_laps[drv_laps["LapTime"].notna()]
    drv_laps["lap_sec"] = drv_laps["LapTime"].dt.total_seconds()
    drv_laps = drv_laps[drv_laps["lap_sec"] > 60]

    # Tyre degradation slope
    # A slope of +0.1 means lap time increases 0.1s per lap on the same tyre set
    deg_slope = 0.0
    if "TyreLife" in drv_laps.columns and len(drv_laps) > 5:
        valid = drv_laps[["TyreLife","lap_sec"]].dropna()
        if len(valid) > 3:
            # np.polyfit fits a straight line — [0] is the slope
            deg_slope = float(np.polyfit(valid["TyreLife"], valid["lap_sec"], 1)[0])

    # Positions gained = grid position minus final position
    grid   = drv_laps.iloc[0].get("GridPosition", 10) if len(drv_laps) > 0 else 10
    finish = drv_laps.iloc[-1].get("Position",    grid) if len(drv_laps) > 0 else grid
    overtakes = max(0, int(grid) - int(finish)) if grid and finish else 0

    info = session.get_driver(driver)
    style_rows.append({
        "driver_number":  driver,
        "driver_id":      info.get("DriverId", str(driver)) if info else str(driver),
        "tyre_deg_slope": round(deg_slope, 4),
        "overtake_count": overtakes,
    })

style_df = pd.DataFrame(style_rows)
print("Driving style features:")
print(style_df.sort_values("tyre_deg_slope").to_string(index=False))

## Step 2.4 — Merge everything into one feature matrix

The final dataset has **one row per driver per race**. Each row contains all features from all three sources. This is the table the model trains on.

The target column `on_podium` is 1 if the driver finished P1/P2/P3, and 0 otherwise. In any race, exactly 3 of 20 rows have `on_podium = 1` — so about 15% of rows are positive.

In [ ]:
# Merge qualifying gap
merged = results_2023.merge(
    quali_2023[["season","round","driver_id","quali_pos","quali_gap"]],
    on=["season","round","driver_id"],
    how="left"
)

# Merge driving style
merged = merged.merge(
    style_df[["driver_id","tyre_deg_slope","overtake_count"]],
    on="driver_id",
    how="left"
)

# Fill remaining gaps
merged["quali_gap"]       = merged["quali_gap"].fillna(3.0)
merged["tyre_deg_slope"]  = merged["tyre_deg_slope"].fillna(0.07)
merged["overtake_count"]  = merged["overtake_count"].fillna(2.0)

FEATURE_COLS = [
    "quali_gap", "quali_pos",
    "rolling_podium_rate", "rolling_win_rate",
    "rolling_points_avg", "rolling_dnf_rate",
    "car_rank", "car_points",
    "tyre_deg_slope", "overtake_count",
]

print(f"Final dataset: {len(merged)} rows")
print(f"Podium rate: {merged['on_podium'].mean()*100:.1f}%")
merged[FEATURE_COLS + ["on_podium"]].describe()

---
# Part 3 — Model training

## Step 3.1 — Train/test split

We split **by season**, not randomly. 2018–2022 goes to training, 2023–2024 to testing.

**Why no random split?** Rows from the same race are highly correlated (all 20 drivers share the same circuit, weather, and safety car conditions). A random split would leak race-level information between train and test, making results look better than they really are. Splitting by season gives a clean temporal boundary — the model truly hasn't seen anything from the test seasons.

In [ ]:
# Load the full multi-season dataset built by f1_podium_features.py
# (or use merged above for a single-season demo)
try:
    full_df = pd.read_csv("f1_podium_features.csv")
    print(f"Loaded full dataset: {len(full_df)} rows")
except FileNotFoundError:
    print("Using single-season demo data.")
    full_df = merged.copy()

full_df = full_df.dropna(subset=["quali_gap", "car_rank", "on_podium"])
full_df[FEATURE_COLS] = full_df[FEATURE_COLS].fillna(0)

train = full_df[full_df["season"] <= 2022]
test  = full_df[full_df["season"] >= 2023]

X_train, y_train = train[FEATURE_COLS], train["on_podium"]
X_test,  y_test  = test[FEATURE_COLS],  test["on_podium"]

print(f"Train: {len(X_train)} rows | Test: {len(X_test)} rows")
print(f"Train podium rate: {y_train.mean()*100:.1f}%")
print(f"Test  podium rate: {y_test.mean()*100:.1f}%")

## Step 3.2 — Train XGBoost with class balancing

**Why XGBoost?**
XGBoost trains decision trees sequentially — each tree learns to fix the errors of all previous trees. This "boosting" approach typically outperforms Random Forest on structured tabular data, especially when features have varying importance (which they do here — `quali_gap` dominates).

**Class imbalance:** Only ~15% of rows are positives (podium finishes). Without correction, the model learns it's safer to always predict "no podium" — it's right 85% of the time.

`scale_pos_weight = negatives / positives` tells XGBoost to penalise missed positives more heavily. If there are 85 non-podium rows for every 15 podium rows, set `scale_pos_weight = 85/15 ≈ 5.7`.

In [ ]:
try:
    import xgboost as xgb
except ImportError:
    from sklearn.ensemble import GradientBoostingClassifier as xgb
    print("xgboost not found — using sklearn GradientBoosting")

neg = (y_train == 0).sum()
pos = (y_train == 1).sum()
scale_weight = neg / pos
print(f"scale_pos_weight = {neg}/{pos} = {scale_weight:.1f}")

model = xgb.XGBClassifier(
    n_estimators=300,
    max_depth=4,              # shallow trees prevent overfitting
    learning_rate=0.05,       # small steps — more trees, each contributing little
    subsample=0.8,            # each tree sees 80% of rows (stochastic GB)
    colsample_bytree=0.8,     # each tree sees 80% of features
    scale_pos_weight=scale_weight,  # fix class imbalance
    use_label_encoder=False,
    eval_metric="logloss",
    random_state=42,
    n_jobs=-1,
)
model.fit(X_train, y_train)
print("Training complete.")

---
# Part 4 — Evaluation

## Step 4.1 — Why accuracy is useless here

A model that always predicts "no podium" would be 85% accurate. That's not useful.

For imbalanced classification, use:
- **ROC-AUC** — how well the model ranks positives above negatives. Range 0.5 (random) to 1.0 (perfect). Target: >0.85.
- **Average precision** — area under the precision-recall curve. Better than ROC-AUC when positives are rare.
- **Top-3 accuracy** — for each race, do the 3 highest probability drivers match the actual podium? This is the real-world metric you care about.

In [ ]:
from sklearn.metrics import (roc_auc_score, average_precision_score,
                              classification_report)

probs  = model.predict_proba(X_test)[:, 1]  # probability of podium
y_pred = (probs >= 0.35).astype(int)         # threshold = 0.35, not 0.5
# Lower threshold because we're trying to catch rare podium events —
# we'd rather have a few false positives than miss actual podiums

print(f"ROC-AUC:           {roc_auc_score(y_test, probs):.3f}")
print(f"Average precision: {average_precision_score(y_test, probs):.3f}")
print()
print(classification_report(y_test, y_pred, target_names=["No podium", "Podium"]))

# Top-3 accuracy per race
test_eval = test.copy()
test_eval["prob"] = probs

correct, total = 0, 0
for (season, rnd), race in test_eval.groupby(["season","round"]):
    if len(race) < 3:
        continue
    predicted = set(race.nlargest(3, "prob")["driver_id"])
    actual    = set(race[race["on_podium"]==1]["driver_id"])
    if len(predicted & actual) >= 2:  # at least 2 of 3 correct
        correct += 1
    total += 1

print(f"Top-3 accuracy: {correct/total*100:.1f}%  ({correct}/{total} races with ≥2 correct)")

## Step 4.2 — Feature importance

Feature importance tells you which inputs the model relied on most.

Expected ranking (roughly):
1. `quali_gap` — qualifying pace is the strongest single pre-race signal
2. `rolling_podium_rate` — recent form captures momentum and consistency
3. `car_rank` — car quality is a ceiling on driver performance
4. `quali_pos` — grid position correlates with race outcome
5. `rolling_points_avg` — consistent points scoring = consistent pace

If your ranking looks very different, check the data pipeline for bugs.

In [ ]:
import matplotlib.pyplot as plt

imp = pd.Series(model.feature_importances_, index=FEATURE_COLS)
imp = imp.sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(8, 5))
imp.plot(kind="barh", ax=ax, color="#7F77DD")
ax.set_xlabel("Feature importance")
ax.set_title("What drives podium predictions?")
ax.spines[["top","right"]].set_visible(False)
plt.tight_layout()
plt.show()

print("\nImportance scores:")
print(imp.sort_values(ascending=False).apply(lambda x: f"{x*100:.1f}%").to_string())

---
# Part 5 — Making predictions

## Step 5.1 — Predict the podium for an upcoming race

Before a race weekend:
1. Fetch qualifying results to get `quali_gap` and `quali_pos`
2. Compute rolling form features from the last 5 races
3. Look up car rank from the constructor standings
4. Run `model.predict_proba()` for all 20 drivers
5. Take the top 3 as your predicted podium

The probability scores tell you how confident the model is — a 85% vs 51% gap between two drivers is much more meaningful than 85% vs 83%.

In [ ]:
import joblib

# Save model
joblib.dump(model, "f1_podium_model.pkl")
print("Model saved to f1_podium_model.pkl")

# Predict for a single race (using test data as example)
EXAMPLE_SEASON = 2023
EXAMPLE_ROUND  = 5   # Miami

example_race = test_eval[
    (test_eval["season"] == EXAMPLE_SEASON) &
    (test_eval["round"]  == EXAMPLE_ROUND)
].copy()

example_race["prob"] = model.predict_proba(example_race[FEATURE_COLS])[:, 1]
example_race = example_race.sort_values("prob", ascending=False)

print(f"\nPredicted podium — 2023 Round {EXAMPLE_ROUND}:")
print(example_race[["driver_id","quali_pos","prob","on_podium"]].head(5).to_string(index=False))
print("\n'on_podium' = 1 means they actually finished on the podium.")

---
# Part 6 — Running the Streamlit dashboard

## Step 6.1 — Launch the UI

Once your model is trained and saved, the dashboard fetches live qualifying data from Jolpica and shows:
- Predicted podium (P1/P2/P3) with confidence scores
- Full field probability ranking
- Feature importance chart
- Per-driver radar chart and stats

```bash
# In your terminal (not in this notebook):
streamlit run f1_podium_dashboard.py
```

This opens a browser window at `http://localhost:8501`. Change the round number in the sidebar to see predictions for different races.

---

# Next steps — how to improve the model

Once the baseline is working, here are the highest-value improvements to try:

**Better features:**
- `circuit_type` encoding (street circuit vs permanent track — affects who benefits from car/style)
- `gap_to_leader_rolling` — how close the driver has been racing to the frontrunners
- `weather_flag` — rain races heavily shuffle the predicted order
- `safety_car_probability` — some circuits historically see more safety cars, which randomises outcomes

**Better modelling:**
- Add cross-validation across seasons (leave-one-season-out) for more robust evaluation
- Try `CatBoost` — handles categorical features (constructor, circuit) natively without encoding
- Calibrate probabilities with `CalibratedClassifierCV` — raw XGBoost probabilities can be overconfident
- Use `Optuna` for hyperparameter tuning instead of guessing values

**Better evaluation:**
- Evaluate separately by circuit type, season, and constructor
- Track calibration: when the model says 80%, does it actually win 80% of the time?
- Build a Brier score tracker across rounds to monitor live performance